# Simple chain tracing

We build our first chain on top of a Databricks serving endpoint, and let MLflow
record every step of it.

The chain has three steps: **preprocess** the question -> **invoke** the LLM -> **postprocess**
the answer to keep only the text content.


### Install the dependencies

In [0]:
%pip install -U --quiet \
    "mlflow[databricks]==3.14.0" \
    "databricks-langchain==0.20.0" \
    "langchain==1.3.14" \
    "langchain-core==1.5.1"

dbutils.library.restartPython()

### Load the shared configuration

Sets the experiment name (`demo_chain`) used by every notebook of this project.

In [0]:
%run ../_config/config_chain

### Turn on automatic tracing

One call, and every chain invocation is logged as a nested trace: one span per link,
with inputs, outputs, latency and token usage. 

In [0]:
import mlflow

mlflow.langchain.autolog()

## Build the chain

`ChatDatabricks` wraps any OpenAI-compatible serving endpoint, so the same code works for a
foundation model, a fine-tuned model or an external model behind a Databricks endpoint.

| Link | Component |
|---|---|
| 1. Preprocess | `RunnableLambda` - normalize the question |
| 2. Prompt | `ChatPromptTemplate` - system persona + user question |
| 3. Invoke | `ChatDatabricks` |
| 4. Postprocess | `StrOutputParser` - keep only the content |

The `|` operator is LCEL: it pipes the output of one link into the next.

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"

SYSTEM_INSTRUCTIONS = (
    "You are a helpful Databricks ML engineer. "
    "Answer the question in a clear and concise manner. "
    "If you don't know the answer, say so - do not make one up."
)


def normalize(inputs: dict) -> dict:
    """Step 1 - collapse whitespace and trim the user question."""
    return {"question": " ".join(inputs["question"].strip().split())}


preprocess = RunnableLambda(normalize).with_config(run_name="preprocess")

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_INSTRUCTIONS),
        ("user", "{question}"),
    ]
)

llm = ChatDatabricks(
    endpoint=ENDPOINT_NAME,
    temperature=0.1,
    max_tokens=512,
)

# Step 3 - StrOutputParser drops the AIMessage envelope and returns the content only.
chain = preprocess | prompt | llm | StrOutputParser()

chain

## Run a batch of questions

Four realistic questions a developer would ask - a smoke test for the endpoint,
and four traces to inspect.

In [0]:
DOCS_DATA = [
    {"question": "what is a scorer in mlflow ?"},
    {"question": "how do I register a model ?"},
]

for item in DOCS_DATA:
    print(chain.invoke(item))
    print("-" * 80)